In [135]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langchain_core.tools import tool, Tool
from pydantic import BaseModel


import subprocess
import sys 

In [136]:

class ExecutePythonScriptInput(BaseModel):
    path: str
    args: list[str] | None = None
    kwargs: dict[str, str | int | float] | None = None


@tool(args_schema=ExecutePythonScriptInput)
def execute_python_script(
    path: str,
    args: list[str] | None = None,
    kwargs: dict[str, str | int | float] | None = None,
) -> str:
    """Run a Python script with positional args and named CLI options.

    Args:
        path: Path to the Python script.
        args: Positional command-line arguments.
        kwargs: Options with values, e.g. {"n": 3} becomes --n 3.
            Keys may include their leading dashes, e.g. --n or -n.
    """
    command = [sys.executable, path, *[str(arg) for arg in (args or [])]]
    for name, value in (kwargs or {}).items():
        option = name if name.startswith("-") else f"--{name}"
        command.extend([option, str(value)])
    result = subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout


@tool
def write_python_script(path: str , content: str):
    "a function to write a python script by specifyign the path and the content"
    with open(path, 'w') as f:
        f.write(content)

In [137]:
chat = ChatOllama(
    model='qwen3.5:4b',
    temperature=0,
    max_tokens=256,
)

SystemPrompt = '''
You are a helpful assistant, the user will ask you to perform some task, you should find the appropriate tool to run the task.
### TOOLS:
- list_python_scripts: is used to list available scripts
- write_python_script: is used to write a specific python script to a specific directory.
- execute_python_script: is used to run a pre-defined python script
'''
tools = [execute_python_script, write_python_script]

messages = [HumanMessage(content='run a python script to print the word Hell after 3 seconds')]
messages_write_python_script = [HumanMessage(content='write a python script that takes two args and returns the sum, put the file in agent_generated_scripts directory and namd it proprely')]

def llm_chat(messages:list, tools: list | None = None) -> list: 
    tools_dict = {tool.name: tool for tool in tools}
    result = chat.bind_tools(tools).invoke([SystemMessage(content=SystemPrompt)] + messages)
    messages.append(result)
    if result.content:
        print(f"[AGENT]: {result.content}")
    if result.content == '' and result.tool_calls:
        for call in result.tool_calls:
            print(f"[TOOL CALL]: {call}")
            tool_call = tools_dict[call['name']]
            tool_result = tool_call.invoke(call)
            print(f"[OBS]: {tool_result.content}")
            messages.append(tool_result)
    return messages

In [138]:
result = llm_chat(messages_write_python_script, tools)

[TOOL CALL]: {'name': 'write_python_script', 'args': {'path': 'agent_generated_scripts/sum_calculator.py', 'content': '#!/usr/bin/env python3\n"""Sum calculator that takes two arguments and returns their sum."""\n\ndef calculate_sum(a, b):\n    """Calculate the sum of two numbers."""\n    return a + b\n\nif __name__ == "__main__":\n    import sys\n    \n    if len(sys.argv) != 3:\n        print("Usage: python sum_calculator.py <num1> <num2>")\n        sys.exit(1)\n    \n    try:\n        num1 = float(sys.argv[1])\n        num2 = float(sys.argv[2])\n        result = calculate_sum(num1, num2)\n        print(f"Sum of {num1} and {num2} is: {result}")\n    except ValueError:\n        print("Error: Please provide two valid numbers as arguments.")\n        sys.exit(1)'}, 'id': '7aaa98ff-f45f-469a-8269-37cc4b9e8982', 'type': 'tool_call'}
[OBS]: null


#### OKAY, we have a primitive (but working) llm agent, now let's add the ASR model:

In [139]:
from qwen_asr import Qwen3ASRModel
import torch

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-0.6B",
    dtype=torch.bfloat16,
    device_map="cuda:0",
    # attn_implementation="flash_attention_2",
    max_inference_batch_size=32, # Batch size limit for inference. -1 means unlimited. Smaller values can help avoid OOM.
    max_new_tokens=256, # Maximum number of tokens to generate. Set a larger value for long audio input.
)



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
## Record 5-second audio
import sounddevice as sd
from scipy.io.wavfile import write

sample_rate = 24000
duration = 5

def record_audio(sr, duration) :
    print(f"Recording audio for {duration} seconds...")
    audio = (sd.rec(
        int(duration * sr),
        samplerate=sr,
        channels=1,
        dtype="int16",
    ), sample_rate)
    sd.wait()
    return audio

In [25]:
%time
results = model.transcribe(
    audio=audio,
    language="English"
)
print(results)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


CPU times: user 0 ns, sys: 14 μs, total: 14 μs
Wall time: 25 μs
[ASRTranscription(language='English', text='Hello, what is the capital of France?', time_stamps=None)]


### Combine audio -> ASR -> LLM -> Output

In [30]:
audio = record_audio(sr=24000, duration=10)
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content= text)]
llm_result = llm_chat(messages=messages, tools=tools)


Recroding audio for 10 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[AGENT]: The "secret of the universe" is something that has fascinated humanity for millennia, but there isn't one single answer that everyone agrees on. Here are a few perspectives:

**From a scientific standpoint:**
- The universe appears to be expanding at an accelerating rate, driven by dark energy
- We're still searching for what constitutes dark matter and dark energy (together they make up about 95% of the universe)
- The fundamental constants of physics seem fine-tuned for life

**From a philosophical standpoint:**
- Some suggest consciousness itself is the secret - that we are the universe experiencing itself
- Others believe the true secret lies in understanding our place within the cosmos and finding meaning in existence

**From an existential perspective:**
- Perhaps the "secret" is simply that we exist at all, and that the act of asking questions is what makes us human

I don't have access to any hidden information or secrets - I'm an AI assistant designed to help with tas

In [31]:
llm_result

[HumanMessage(content='What is the secret of the universe that no one else knows, and you can tell me it right now without any hesitation, please?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The "secret of the universe" is something that has fascinated humanity for millennia, but there isn\'t one single answer that everyone agrees on. Here are a few perspectives:\n\n**From a scientific standpoint:**\n- The universe appears to be expanding at an accelerating rate, driven by dark energy\n- We\'re still searching for what constitutes dark matter and dark energy (together they make up about 95% of the universe)\n- The fundamental constants of physics seem fine-tuned for life\n\n**From a philosophical standpoint:**\n- Some suggest consciousness itself is the secret - that we are the universe experiencing itself\n- Others believe the true secret lies in understanding our place within the cosmos and finding meaning in existence\n\n**From an existential perspective:**\n-

### The Pipeline looks fairly good, we will now deal with voice verification 

In [6]:
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier

classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
signal, fs =torchaudio.load('recording.wav')
print(signal.shape)
signal = signal.mean(dim=0, keepdim=True)
signal = torchaudio.functional.resample(signal, fs, 16000) # necessary for ecapa-tdnn
embeddings = classifier.encode_batch(signal)


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


torch.Size([1, 120000])


In [5]:
from speechbrain.inference.speaker import SpeakerRecognition
verification = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="pretrained_models/spkrec-ecapa-voxceleb")
score, prediction = verification.verify_files("recording.wav", "recording_dif.wav") # Different Speakers
print(f"{score= }, {prediction= }")
score, prediction = verification.verify_files("recording.wav", "recording_sim.wav") # Same Speaker
print(f"{score= }, {prediction= }")

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


score= tensor([0.2358], device='cuda:0'), prediction= tensor([False], device='cuda:0')
score= tensor([0.7795], device='cuda:0'), prediction= tensor([True], device='cuda:0')


In [234]:

import torch
import torch.nn.functional as F 


def enroll_speaker(trials=2, trial_duration=5):
    pass_threshold = 0.7
    print(f"You will be asked to speak {trials} times, each time you can say whatever you want for {trial_duration} seconds, the model will save your voiceprint afterwards")
    embeddings = []
    for i in range(trials):
        input(f"Press Enter to Start trial {i+1}...")
        print(f"Trial number {i+1} : ")
        audio, sr = record_audio(16000, trial_duration)
        audio = torch.tensor(audio, dtype=torch.float32)
        audio = audio.T
        audio = audio.mean(dim=0, keepdim=True) # in case of multiple channels, ecapa accepts mono audio with sr = 16000
        audio = torchaudio.functional.resample(audio, sr, 16000)
        embeddings.append(classifier.encode_batch(audio, normalize=False))
        if i > 0:
            similarity = F.cosine_similarity(embeddings[-1][0,:,:], embeddings[-2][0,:,:])
            assert similarity.item() > pass_threshold, "Your recorded voice is very different from the previous, try again"
    res = torch.cat(embeddings, dim=0)
    res = F.normalize(res, p=2, dim=-1)
    print(f"Finished Enrollment Procedure")
    return F.normalize(res.mean(dim=0), dim=-1)



def authorized_speaker(audio, sr, ref_dim:torch.tensor) -> bool:
    """Checks the similarity with the enrolled speaker """
    if not isinstance(audio, torch.Tensor):
        audio = torch.tensor(audio, dtype=torch.float32)
        if audio.shape[0] != 1:
            audio = audio.T
    print(audio.shape)
    audio = audio.mean(dim=0, keepdim=True)
    audio = torchaudio.functional.resample(audio, sr, 16000)
    audio_dim = classifier.encode_batch(audio, normalize=False)[0]
    audio_dim = F.normalize(audio_dim, p=2, dim=-1)
    sim = F.cosine_similarity(audio_dim, ref_dim)
    print(f"{sim=}")
    return sim.item() > 0.8




In [248]:
authorized_spk_dim = enroll_speaker(trials=3, trial_duration=5)

You will be asked to speak 3 times, each time you can say whatever you want for 5 seconds, the model will save your voiceprint afterwards
Trial number 1 : 
Recording audio for 5 seconds...
Trial number 2 : 
Recording audio for 5 seconds...


AssertionError: Your recorded voice is very different from the previous, try again

In [245]:
audio = record_audio(16000, 5)

Recording audio for 5 seconds...


In [247]:
authorized_speaker(audio[0], audio[1], authorized_spk_dim)

torch.Size([1, 120000])
sim=tensor([0.5490], device='cuda:0')


False

### The speaker verification is working fine for now (needs further testing to check the authorization).
### The next step is to add it to the pipeline : Audio -> Speaker authorization -> ASR -> LLM -> Response

In [149]:
audio = record_audio(16000, 10)
if not authorized_speaker(audio[0], audio[1], authorized_spk_dim):
    raise Exception('Unauthorized speaker.')
text = model.transcribe(audio=audio, language="English")[0].text
messages = [HumanMessage(content=text)]
tools = [execute_python_script, write_python_script]
response = llm_chat(messages=messages, tools=tools)

Recording audio for 10 seconds...


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


torch.Size([1, 240000])
tensor([0.8576], device='cuda:0')


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[TOOL CALL]: {'name': 'write_python_script', 'args': {'path': 'multiply_three_numbers.py', 'content': '#!/usr/bin/env python3\n"""Script to multiply three numbers together."""\n\ndef multiply_three_numbers(num1, num2, num3):\n    """Multiply three numbers and return the result."""\n    result = num1 * num2 * num3\n    return result\n\nif __name__ == "__main__":\n    import sys\n    \n    # Get three numbers from command line arguments\n    if len(sys.argv) != 4:\n        print("Usage: python multiply_three_numbers.py <num1> <num2> <num3>")\n        sys.exit(1)\n    \n    try:\n        num1 = float(sys.argv[1])\n        num2 = float(sys.argv[2])\n        num3 = float(sys.argv[3])\n        \n        result = multiply_three_numbers(num1, num2, num3)\n        print(f"{num1} × {num2} × {num3} = {result}")\n    except ValueError:\n        print("Error: Please provide three valid numbers.")\n        sys.exit(1)\n'}, 'id': '83852c3e-6979-40ec-a1fb-56c0a6019540', 'type': 'tool_call'}
[OBS]: nul

Go ahead and write a script Python that takes two arguments and returns their sum with a square root.

null


### 